In [ ]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk16.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:677: RuntimeWarning: Optimization failed in `gen_candidates_scipy` with the following warning(s):
[OptimizationWarning('Optimization failed within `scipy.optimize.minimize` with status 2 and message ABNORMAL: .')]
Trying again with a new set of initial conditions.
  return _optimize_acqf_batch(opt_inputs=opt_inputs)


In [3]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(7): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=3)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
17.75041401910035

Trial 1 =========================================
17.765223898892707

Trial 2 =========================================
16.867351329854266

Trial 3 =========================================
17.816807095240144

Trial 4 =========================================
18.00664541328186

Trial 5 =========================================
18.156929781894974

Trial 6 =========================================
17.87723106387071



/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 7 =========================================
14.424650742871801

Trial 8 =========================================
17.806444305929134

Trial 9 =========================================
17.738514617220893

Trial 10 =========================================
17.84737128283801

Trial 11 =========================================
17.77584314012978

Trial 12 =========================================
17.82667715606422

Trial 13 =========================================
17.41151329345884

Trial 14 =========================================
17.77942265344643

Trial 15 =========================================
17.82392925413115

Trial 16 =========================================
17.826019298564322

Trial 17 =========================================
17.63877225226312

Trial 18 =========================================
17.925767062354364

Trial 19 =========================================
17.776782511837585

Trial 20 =========================================
17.655884510004398



/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 21 =========================================
17.901408549284945

Trial 22 =========================================
17.88874353191023

Trial 23 =========================================
17.891104362515335

Trial 24 =========================================
17.975981302227318

Trial 25 =========================================
17.56622500740368

Trial 26 =========================================
17.79537117328681

Trial 27 =========================================
17.701475085461233

Trial 28 =========================================
17.65276866717587



...

Trial 75 =========================================
14.417400256170202

Trial 76 =========================================
17.682230058323285

Trial 77 =========================================
17.97423871475167

Trial 78 =========================================
17.562145715507015

Trial 79 =========================================
18.029246711936832

Trial 80 =========================================
18.08028906953028

Trial 81 =========================================
18.056938056972346

Trial 82 =========================================
17.73057591849871



/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 83 =========================================
17.782024155662807

Trial 84 =========================================
17.801805813572713

Trial 85 =========================================
17.981728281766905

Trial 86 =========================================
18.02713121979159

Trial 87 =========================================
18.005396514666444

Trial 88 =========================================
17.564666080479917

Trial 89 =========================================
17.954820349538352

Trial 90 =========================================
17.766045864326525

Trial 91 =========================================
17.73423340347702

Trial 92 =========================================
17.941124795486303



/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 93 =========================================
17.762650042395812

Trial 94 =========================================
17.74070674222294

Trial 95 =========================================
18.11906556522107

Trial 96 =========================================
17.88879086532046

Trial 97 =========================================
17.876742507073356

Trial 98 =========================================
17.949752847098008

Trial 99 =========================================
17.64525588681857

[17.75041402 17.7652239  16.86735133 17.8168071  18.00664541 18.15692978
 17.87723106 14.42465074 17.80644431 17.73851462 17.84737128 17.77584314
 17.82667716 17.41151329 17.77942265 17.82392925 17.8260193  17.63877225
 17.92576706 17.77678251 17.65588451 17.90140855 17.88874353 17.89110436
 17.9759813  17.56622501 17.79537117 17.70147509 17.65276867 17.76370221
 17.923349   17.93769121 17.73450288 18.03812819 17.65735187 18.0528042
 17.96102662 17.86433954 18.0487456  17.86703056 17.76205031 18.062233

In [4]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.156929781894974
Avg = 17.73757690601447
Std = 0.5670813501935181


In [5]:
print(y_max_arr.tolist())

[17.75041401910035, 17.765223898892707, 16.867351329854266, 17.816807095240144, 18.00664541328186, 18.156929781894974, 17.87723106387071, 14.424650742871801, 17.806444305929134, 17.738514617220893, 17.84737128283801, 17.77584314012978, 17.82667715606422, 17.41151329345884, 17.77942265344643, 17.82392925413115, 17.826019298564322, 17.63877225226312, 17.925767062354364, 17.776782511837585, 17.655884510004398, 17.901408549284945, 17.88874353191023, 17.891104362515335, 17.975981302227318, 17.56622500740368, 17.79537117328681, 17.701475085461233, 17.65276866717587, 17.763702214693783, 17.923348998409875, 17.937691209115133, 17.734502876266816, 18.038128192752794, 17.657351865117104, 18.05280420484229, 17.961026619452554, 17.86433954472423, 18.048745602599, 17.86703056401817, 17.76205030890539, 18.06223393020946, 18.112610558995314, 17.801585291654227, 18.083358335598017, 17.684279327533673, 17.61008230289356, 17.783407824895733, 17.14537513167682, 17.806961196402032, 15.396636170068561, 17.

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestswGPModel/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/SequentialTestswGPModel/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    17.982576
1    17.830784
2    18.091045
3    17.677428
4    17.875698
..         ...
595  18.119066
596  17.888791
597  17.876743
598  17.949753
599  17.645256

[600 rows x 1 columns]
